# Training an Email Triage Classifier — Explained

This notebook walks through every stage of training a **multi-label** email classifier, from raw text to a saved checkpoint. Each section explains *why* a technique is used, then demonstrates it with runnable code.

**What you'll learn:**
- Why multi-label classification needs different tools than multi-class
- How emails become tensors (tokenization + multi-hot label vectors)
- The three-layer model architecture (backbone → pooling → head)
- Why we use `BCEWithLogitsLoss` instead of `CrossEntropyLoss`
- How the training loop, metrics, checkpointing, and MLflow tracking fit together

All code lives in `src/email_triage/` — this notebook **imports and demonstrates** the existing modules rather than reimplementing them.

In [ ]:
import sys
from pathlib import Path

# Make `email_triage` importable from the notebook
ROOT = Path.cwd().parent
SRC = str(ROOT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import numpy as np
import torch
import torch.nn as nn

print(f"PyTorch {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

---
## 1 — Multi-Label vs Multi-Class

In **multi-class** classification (e.g. "is this a cat, dog, or bird?"), exactly one label is correct, so we use **softmax** — probabilities that sum to 1 across all classes.

Email triage is **multi-label**: a single email can be `urgent` *and* `needs_reply` *and* `action_required` all at once. Each label is an **independent** binary decision (yes/no), so we use **sigmoid** on each output independently. The probabilities do **not** need to sum to 1.

In [ ]:
# Example emails showing why multi-label matters
examples = [
    {
        "subject": "Your Amazon order has shipped",
        "labels": ["order_confirmation"],
    },
    {
        "subject": "URGENT: Server CPU at 98% — please investigate",
        "labels": ["urgent", "action_required", "alerts"],
    },
    {
        "subject": "Team standup moved to 3pm — RSVP needed",
        "labels": ["needs_reply", "calendar_event"],
    },
]

for ex in examples:
    print(f"Subject: {ex['subject']}")
    print(f"  Labels: {ex['labels']}")
    print()

---
## 2 — Label Taxonomy & Binary Vectors

Neural networks need fixed-length numeric inputs and outputs. We represent labels as a **multi-hot binary vector** — a list of 7 values, one per label, where `1` means the label is active.

This is defined in `src/email_triage/labels.py`.

In [ ]:
from email_triage.labels import LABELS, LABEL_NAMES, NUM_LABELS

print(f"Number of labels: {NUM_LABELS}\n")
for i, label in enumerate(LABELS):
    print(f"  [{i}] {label.name:20s} — {label.description}")

In [ ]:
from email_triage.labels import label_vector, vector_to_labels

# Encode: label names -> binary vector
active = ["urgent", "alerts"]
vec = label_vector(active)
print(f"label_vector({active})")
print(f"  -> {vec}")
print(f"     {'  '.join(LABEL_NAMES)}")
print()

# Decode: binary vector -> label names
recovered = vector_to_labels(vec)
print(f"vector_to_labels({vec})")
print(f"  -> {recovered}")
print()

# Threshold behaviour with probabilities
probs = [0.9, 0.3, 0.1, 0.0, 0.8, 0.45, 0.6]
print(f"vector_to_labels({probs}, threshold=0.5)")
print(f"  -> {vector_to_labels(probs, threshold=0.5)}")

---
## 3 — Tokenization

Transformer models can't read raw text — they need **token IDs** (integers from a fixed vocabulary). The tokenizer also produces an **attention mask** that tells the model which positions are real tokens vs padding.

Our `EmailDataset` concatenates the subject and body with a `[SEP]` separator: `"subject [SEP] body"`, then tokenizes the result.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

from email_triage.model.classifier import build_tokenizer

tokenizer = build_tokenizer(MODEL_NAME)

text = "URGENT: Server down [SEP] The production database is unreachable. Please investigate immediately."
encoding = tokenizer(text, padding="max_length", truncation=True, max_length=32, return_tensors="pt")

print("Input text:")
print(f"  {text}\n")

print("Token IDs (first 20):")
ids = encoding["input_ids"][0]
print(f"  {ids[:20].tolist()}\n")

print("Decoded tokens (first 20):")
tokens = tokenizer.convert_ids_to_tokens(ids[:20])
print(f"  {tokens}\n")

print("Attention mask (first 20):")
print(f"  {encoding['attention_mask'][0][:20].tolist()}")
print(f"  (1 = real token, 0 = padding)")

`EmailDataset` (in `src/email_triage/data/dataset.py`) wraps this into a PyTorch `Dataset`. Each `__getitem__` call returns a dict with three tensors:

```python
{
    "input_ids":      tensor of shape (max_length,),
    "attention_mask": tensor of shape (max_length,),
    "labels":         tensor of shape (7,),  # multi-hot binary vector
}
```

The `DataLoader` then batches these into `(B, max_length)` and `(B, 7)` tensors.

---
## 4 — Model Architecture

The `EmailTriageClassifier` has three layers:

1. **Backbone** — a pretrained DistilBERT that converts token IDs into contextual embeddings (768-dim vectors per token)
2. **Pooling** — reduces the sequence of token embeddings into a single vector. CLS pooling takes the first token; mean pooling averages all non-padding tokens.
3. **Classification head** — a single `nn.Linear(768, 7)` layer that maps the pooled vector to 7 raw logits (one per label)

The backbone has ~66M parameters that were pretrained on massive text corpora. The classification head adds only `768 × 7 + 7 = 5,383` parameters. Fine-tuning updates *all* of them.

In [ ]:
from email_triage.model.classifier import build_classifier

model = build_classifier(MODEL_NAME)

# High-level structure
print("Top-level modules:")
for name, child in model.named_children():
    n_params = sum(p.numel() for p in child.parameters())
    print(f"  {name:12s}  {n_params:>12,} params")

total = sum(p.numel() for p in model.parameters())
print(f"{'':12s}  {'─' * 18}")
print(f"  {'total':12s}  {total:>12,} params")

### CLS pooling vs Mean pooling

The `_pool()` method in `EmailTriageClassifier` supports two strategies:

- **CLS pooling** (`pooling="cls"`): Takes the hidden state at position 0 (the `[CLS]` token). BERT-style models are pretrained so this position captures a sentence-level summary.

  ```python
  hidden_states[:, 0, :]          # shape: (B, H)
  ```

- **Mean pooling** (`pooling="mean"`): Averages all non-padding hidden states, giving every token equal weight.

  ```python
  mask = attention_mask.unsqueeze(-1).float()    # (B, L, 1)
  summed = (hidden_states * mask).sum(dim=1)     # (B, H)
  summed / mask.sum(dim=1).clamp(min=1e-9)       # (B, H)
  ```

CLS pooling is the default and works well for classification tasks.

In [ ]:
# Trace the forward pass shape-by-shape
model.eval()

sample = tokenizer("Meeting tomorrow at 10am", padding="max_length", truncation=True,
                    max_length=256, return_tensors="pt")

with torch.no_grad():
    # Step 1: Backbone
    backbone_out = model.backbone(
        input_ids=sample["input_ids"],
        attention_mask=sample["attention_mask"],
    )
    hidden = backbone_out.last_hidden_state
    print(f"1. Backbone output:  {hidden.shape}   (batch, seq_len, hidden_dim)")

    # Step 2: Pooling
    pooled = model._pool(hidden, sample["attention_mask"])
    print(f"2. After pooling:    {pooled.shape}         (batch, hidden_dim)")

    # Step 3: Classification head
    logits = model.classifier(model.dropout(pooled))
    print(f"3. Logits:           {logits.shape}            (batch, num_labels)")

    # Step 4: Sigmoid -> probabilities
    probs = torch.sigmoid(logits)
    print(f"4. Probabilities:    {probs.shape}            (batch, num_labels)")
    print()
    print("Label probabilities:")
    for name, p in zip(LABEL_NAMES, probs[0]):
        bar = "█" * int(p.item() * 40)
        print(f"  {name:20s} {p.item():.4f} {bar}")

---
## 5 — Loss Function: BCEWithLogitsLoss

For multi-class tasks you'd use `CrossEntropyLoss` (which applies softmax internally). But here each label is an **independent** binary classification, so we need **Binary Cross-Entropy**.

`BCEWithLogitsLoss` combines sigmoid + BCE in a single numerically stable operation:

$$\mathcal{L} = -\frac{1}{N \cdot C} \sum_{i=1}^{N} \sum_{j=1}^{C} \Big[ y_{ij} \log(\sigma(z_{ij})) + (1 - y_{ij}) \log(1 - \sigma(z_{ij})) \Big]$$

Where $z$ are the raw logits (before sigmoid) and $\sigma$ is the sigmoid function. The key insight: **it operates independently on each of the 7 outputs**, treating each as its own binary classification problem.

In [ ]:
# Demonstrate BCE loss on synthetic data
criterion = nn.BCEWithLogitsLoss(reduction="none")

# Fake logits from the model (before sigmoid)
logits = torch.tensor([[ 2.0, -1.5,  0.3,  0.0, -3.0,  1.0,  0.8]])
# Ground truth: urgent + action_required + calendar_event
targets = torch.tensor([[1.0,  0.0,  1.0,  0.0,  0.0,  1.0,  0.0]])

per_label_loss = criterion(logits, targets)

print("Per-label loss breakdown:")
print(f"{'Label':20s} {'Logit':>7s} {'P(y=1)':>7s} {'Target':>7s} {'Loss':>7s}")
print("-" * 50)
for name, l, p, t, loss_val in zip(
    LABEL_NAMES,
    logits[0],
    torch.sigmoid(logits[0]),
    targets[0],
    per_label_loss[0],
):
    print(f"{name:20s} {l.item():7.2f} {p.item():7.4f} {t.item():7.0f} {loss_val.item():7.4f}")

print(f"\nMean loss: {per_label_loss.mean().item():.4f}")

In [ ]:
# The sigmoid mapping: logit -> probability
logit_range = torch.linspace(-6, 6, 13)

print("Sigmoid mapping (logit -> probability):")
print(f"{'Logit':>7s}  {'Prob':>7s}  {'Predicted':>10s}")
print("-" * 30)
for z in logit_range:
    p = torch.sigmoid(z).item()
    pred = "YES" if p >= 0.5 else "no"
    print(f"{z.item():7.1f}  {p:7.4f}  {pred:>10s}")

print("\nKey insight: logit=0 maps to probability=0.5 (the decision boundary)")

---
## 6 — Training Loop Mechanics

Each training step follows four stages:

1. **Forward pass** — feed the batch through the model to get logits
2. **Loss** — compare logits to ground-truth labels with `BCEWithLogitsLoss`
3. **Backward pass** — `loss.backward()` computes gradients for every parameter
4. **Optimizer step** — `optimizer.step()` updates parameters using the gradients

We use **AdamW** (Adam with decoupled weight decay), which adapts the learning rate per-parameter based on gradient history. The starting learning rate `2e-5` is a standard choice for fine-tuning pretrained transformers — large enough to learn, small enough not to destroy pretrained features.

In [ ]:
from torch.optim import AdamW

# Inspect optimizer param groups (using the real model)
optimizer = AdamW(model.parameters(), lr=2e-5)

for i, group in enumerate(optimizer.param_groups):
    n_params = sum(p.numel() for p in group["params"])
    print(f"Param group {i}:")
    print(f"  lr           = {group['lr']}")
    print(f"  weight_decay = {group['weight_decay']}")
    print(f"  # parameters = {n_params:,}")

In [ ]:
# Simulate one training step on a tiny model (no heavy transformer needed)
tiny_model = nn.Linear(4, 3)  # 4 inputs, 3 labels
tiny_optimizer = AdamW(tiny_model.parameters(), lr=0.01)
tiny_criterion = nn.BCEWithLogitsLoss()

# Fake batch: 2 samples, 4 features
x = torch.randn(2, 4)
y = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0]])

print("Before training step:")
print(f"  weights = {tiny_model.weight.data[0].tolist()}")

# Step 1: Forward
logits = tiny_model(x)
print(f"  logits  = {logits[0].tolist()}")

# Step 2: Loss
loss = tiny_criterion(logits, y)
print(f"  loss    = {loss.item():.4f}")

# Step 3: Backward
tiny_optimizer.zero_grad()
loss.backward()
print(f"  grads   = {tiny_model.weight.grad[0].tolist()}")

# Step 4: Optimizer step
tiny_optimizer.step()

print("\nAfter training step:")
print(f"  weights = {tiny_model.weight.data[0].tolist()}")
print("\n  (Weights moved in the direction that reduces the loss)")

### Gradient Clipping

Fine-tuning pretrained models can sometimes produce very large gradients ("gradient explosion"), especially early in training. **Gradient clipping** caps the total gradient norm to prevent destructive updates.

From `trainer.py`:
```python
nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)  # default: 1.0
```

This rescales all gradients proportionally so the combined norm never exceeds `max_grad_norm`. Individual gradient directions are preserved — only the magnitude is capped.

---
## 7 — The `_run_epoch` Function

The `_run_epoch` function in `trainer.py` handles both training and evaluation in a single function. Here's the logic:

```python
def _run_epoch(model, loader, criterion, device, optimizer=None, max_grad_norm=1.0):
```

**Dual mode:** If `optimizer` is passed → train mode; if `None` → eval mode.

1. Set `model.train()` or `model.eval()` (controls dropout & batch norm behavior)
2. Wrap in `torch.no_grad()` for eval (saves memory, skips gradient computation)
3. For each batch:
   - Move tensors to device (CPU/GPU)
   - Forward pass → logits → loss
   - **If training:** zero gradients → backward → clip → optimizer step
   - Accumulate loss and collect sigmoid probabilities
4. Return `(avg_loss, y_true, y_probs)` as numpy arrays

This design avoids code duplication — the same function runs train epochs, validation epochs, and final evaluation.

---
## 8 — Metrics

For multi-label classification, accuracy is misleading (predicting all zeros scores well when labels are sparse). Instead we use **precision**, **recall**, and **F1-score**:

- **Precision** = of all predicted positives, how many are correct?
- **Recall** = of all actual positives, how many did we find?
- **F1** = harmonic mean of precision and recall

Two averaging strategies:
- **Micro** — pool all labels together, then compute. Gives more weight to frequent labels.
- **Macro** — compute per-label, then average. Treats every label equally regardless of frequency.

We use **macro-F1** for model selection (checkpointing) because we want the model to perform well on *all* labels, not just the common ones.

In [ ]:
from email_triage.training.metrics import compute_metrics

# Synthetic ground truth and predictions for 5 samples
y_true = np.array([
    [1, 0, 1, 0, 0, 0, 0],  # urgent + action_required
    [0, 1, 0, 0, 0, 1, 0],  # needs_reply + calendar_event
    [1, 1, 1, 0, 1, 0, 0],  # urgent + needs_reply + action_required + alerts
    [0, 0, 0, 1, 0, 0, 0],  # order_confirmation
    [0, 0, 0, 0, 0, 0, 1],  # newsletters
])

y_pred = np.array([
    [1, 0, 1, 0, 0, 0, 0],  # correct
    [0, 1, 0, 0, 0, 0, 0],  # missed calendar_event
    [1, 0, 1, 0, 1, 0, 0],  # missed needs_reply
    [0, 0, 0, 1, 0, 0, 0],  # correct
    [0, 0, 0, 0, 0, 0, 1],  # correct
])

result = compute_metrics(y_true, y_pred)

print(f"Macro F1: {result.macro_f1:.4f}")
print(f"Micro F1: {result.micro_f1:.4f}")
print()
print(f"{'Label':20s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s}")
print("-" * 52)
for name in LABEL_NAMES:
    m = result.per_label[name]
    print(f"{name:20s} {m['precision']:10.4f} {m['recall']:10.4f} {m['f1']:10.4f}")

In [ ]:
# Full classification report (scikit-learn format)
print(result.report_text)

---
## 9 — Checkpointing

Training runs for multiple epochs, and the model doesn't always improve monotonically — later epochs may overfit or just get unlucky with a batch. So we save the **best** checkpoint (by macro-F1), not the last one.

The checkpoint dict contains everything needed to resume or deploy:

```python
torch.save({
    "epoch":                epoch,
    "model_state_dict":     model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config":               asdict(config),
    "val_macro_f1":         best_macro_f1,
}, checkpoint_path)
```

- `model_state_dict` — all parameter values (weights + biases)
- `optimizer_state_dict` — AdamW's internal momentum/velocity buffers (needed to resume training)
- `config` — hyperparameters used (for reproducibility)
- `val_macro_f1` — the score that justified saving this checkpoint

In [ ]:
# Inspect a checkpoint if one exists
checkpoint_path = ROOT / "models" / "best.pt"

if checkpoint_path.exists():
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    print(f"Checkpoint keys: {list(ckpt.keys())}")
    print(f"Saved at epoch:  {ckpt['epoch']}")
    print(f"Val macro F1:    {ckpt['val_macro_f1']:.4f}")
    print(f"Config:          {ckpt['config']}")
    print(f"\nModel state dict: {len(ckpt['model_state_dict'])} parameter tensors")
else:
    print(f"No checkpoint found at {checkpoint_path}")
    print("(This is expected if you haven't trained the model yet)")
    print()
    # Show what the state dict looks like from the current model
    sd = model.state_dict()
    print(f"Current model state dict has {len(sd)} parameter tensors.")
    print("First 5 keys:")
    for key in list(sd.keys())[:5]:
        print(f"  {key:50s} shape={sd[key].shape}")

---
## 10 — MLflow Tracking

MLflow logs every training run so we can compare experiments later. Our `train()` function logs:

- **Parameters** — all fields from `TrainConfig` (model name, lr, epochs, batch size, etc.)
- **Metrics per epoch** — `train_loss`, `val_loss`, `val_macro_f1`, `val_micro_f1`, and per-label F1 scores
- **Final metric** — `best_val_macro_f1`

Everything is stored in a local SQLite database at `outputs/mlflow.db`.

In [ ]:
import mlflow

tracking_uri = f"sqlite:///{ROOT / 'outputs' / 'mlflow.db'}"
db_path = ROOT / "outputs" / "mlflow.db"

if db_path.exists():
    mlflow.set_tracking_uri(tracking_uri)
    runs = mlflow.search_runs(order_by=["start_time DESC"])
    if len(runs) > 0:
        print(f"Found {len(runs)} run(s). Most recent:")
        cols = [c for c in ["run_id", "status", "metrics.best_val_macro_f1",
                            "params.model_name", "params.lr", "params.epochs"]
                if c in runs.columns]
        print(runs[cols].head().to_string(index=False))
    else:
        print("MLflow database exists but contains no runs yet.")
else:
    print(f"No MLflow database found at {db_path}")
    print("(It will be created automatically on the first training run)")

### Launching the MLflow UI

To browse runs interactively, start the MLflow UI from the project root:

```bash
mlflow ui --backend-store-uri sqlite:///outputs/mlflow.db --port 5000
```

Then open [http://localhost:5000](http://localhost:5000) in your browser. You can compare runs, view metric curves, and inspect logged parameters.

---
## 11 — Putting It All Together

Here's how every concept maps to the `train()` function in `trainer.py`:

| Concept | Where it appears |
|---|---|
| Multi-hot label vectors | `EmailDataset.__getitem__` → `label_vector()` |
| Tokenization | `EmailDataset.__getitem__` → `tokenizer(...)` |
| Model architecture | `EmailTriageClassifier` (backbone + pooling + head) |
| BCEWithLogitsLoss | `criterion = nn.BCEWithLogitsLoss()` |
| AdamW optimizer | `optimizer = AdamW(model.parameters(), lr=config.lr)` |
| Forward + backward + step | `_run_epoch()` with `optimizer` argument |
| Gradient clipping | `nn.utils.clip_grad_norm_(...)` inside `_run_epoch()` |
| Evaluation metrics | `evaluate()` → `compute_metrics()` → `EvalResult` |
| Best checkpoint | `if val_result.macro_f1 > best_macro_f1: torch.save(...)` |
| MLflow logging | `mlflow.log_params(...)` + `mlflow.log_metrics(...)` per epoch |

The training loop runs for `config.epochs` iterations. Each epoch does one full pass through the training data, followed by evaluation on the validation set. The best checkpoint (by macro-F1) is saved and reloaded at the end.

In [ ]:
# Complete training call using the real modules.
# Set RUN_TRAINING = True to actually train (will take a while!).

RUN_TRAINING = False  # Safety guard — flip to True when you're ready

if RUN_TRAINING:
    from torch.utils.data import DataLoader

    from email_triage.data.dataset import EmailDataset
    from email_triage.model.classifier import build_classifier, build_tokenizer
    from email_triage.training import TrainConfig, train

    config = TrainConfig()
    tok = build_tokenizer(config.model_name)
    clf = build_classifier(config.model_name)

    train_ds = EmailDataset(f"{config.data_dir}/train.jsonl", tok)
    val_ds = EmailDataset(f"{config.data_dir}/val.jsonl", tok)

    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size)

    trained_model = train(clf, train_loader, val_loader, config)
    print("Done!")
else:
    print("Skipping training (RUN_TRAINING = False).")
    print("Set RUN_TRAINING = True in this cell to run the full training loop.")